# EcoBridge offline e-waste classifier
Run this notebook in Google Colab after uploading `datasets/ai_ml/vision` as `vision.zip`. It trains the eight-class on-device classifier and exports the exact assets consumed by `ScrapClassifier`. This small dataset is a demo baseline; collect substantially more labelled field images before production use.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload vision.zip
!rm -rf /content/vision /content/ecobridge_dataset
!unzip -q vision.zip -d /content/vision
import pathlib, shutil
root = pathlib.Path('/content/vision')
if not (root / 'train_classes').exists():
    root = next(path for path in root.rglob('vision') if (path / 'train_classes').exists())
target = pathlib.Path('/content/ecobridge_dataset')
for split, source in [('train', 'train_classes'), ('validation', 'val_classes')]:
    for category in (root / source).iterdir():
        if category.is_dir(): shutil.copytree(category, target / split / category.name, dirs_exist_ok=True)
assert len(list((target / 'train').iterdir())) == 8, 'Expected eight folders under train_classes'


In [ ]:
import tensorflow as tf
image_size, batch_size = (224, 224), 8
train = tf.keras.utils.image_dataset_from_directory('/content/ecobridge_dataset/train', image_size=image_size, batch_size=batch_size, label_mode='int')
validation = tf.keras.utils.image_dataset_from_directory('/content/ecobridge_dataset/validation', image_size=image_size, batch_size=batch_size, label_mode='int', shuffle=False)
labels = train.class_names
augmentation = tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'), tf.keras.layers.RandomRotation(.08), tf.keras.layers.RandomContrast(.1)])
base = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base.trainable = False
model = tf.keras.Sequential([augmentation, tf.keras.applications.mobilenet_v2.preprocess_input, base, tf.keras.layers.GlobalAveragePooling2D(), tf.keras.layers.Dropout(.2), tf.keras.layers.Dense(len(labels), activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train.prefetch(tf.data.AUTOTUNE), validation_data=validation.prefetch(tf.data.AUTOTUNE), epochs=20, callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])


In [ ]:
# Keep float input/output: Android's current interpreter sends normalized float RGB.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
open('mobilenet_scrap_v1.tflite', 'wb').write(converter.convert())
assert labels == ['BATTERIES', 'CABLES', 'CRT', 'LCD_LED_PANEL', 'MIXED_PLASTICS', 'MOTORS_MAGNETS', 'OTHER_EWASTE', 'PCB'], labels
label_map = {'BATTERIES': 'Batteries', 'CABLES': 'Copper Cables & Wires', 'CRT': 'CRT Monitors & TVs', 'LCD_LED_PANEL': 'LCD / LED Panels', 'MIXED_PLASTICS': 'Mixed E-Waste Plastics', 'MOTORS_MAGNETS': 'Motors & Magnet Assemblies', 'OTHER_EWASTE': 'Other Electronic Scrap', 'PCB': 'Printed Circuit Boards (PCBs)'}
open('labels.txt', 'w').write('\n'.join(label_map[label] for label in labels) + '\n')
files.download('mobilenet_scrap_v1.tflite'); files.download('labels.txt')
# Copy both downloaded files to collector_app/app/src/main/assets/ and build the APK.
